In [15]:
import os
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re


In [3]:
#checking if it is loading or not
pdf_folder = "papers"

pdf_files = [file for file in os.listdir(pdf_folder)]

for file in pdf_files:

    path = os.path.join(pdf_folder, file)

    pdf = PdfReader(path)

    text = ""

    for page in pdf.pages:
        extracted = page.extract_text()

        if extracted:
            text += extracted

    print("=" * 50)
    print("File:", file)
    print("Pages:", len(pdf.pages))
    print("Characters:", len(text))

File: 2025_AnnualReport.pdf
Pages: 80
Characters: 251108
File: rbireport.pdf
Pages: 246
Characters: 688435
File: Revolutionizing_Fashion_Retail_Virtual_Try-on_Empowered_By_Deep_Learning.pdf
Pages: 6
Characters: 24544


In [8]:
#fixed chunking

pdf1 = PdfReader("papers/Revolutionizing_Fashion_Retail_Virtual_Try-on_Empowered_By_Deep_Learning.pdf")

text = ""

for page in pdf.pages:
    extracted = page.extract_text()

    if extracted:
        text += extracted


overlap = 50
chunksize= 500

chunks = []

start = 0

while start < len(text):
    end = start + chunksize

    chunk = text[start:end]
    chunks.append(chunk)
    
    start += chunksize - overlap
    
print("Total Chunks:", len(chunks))

chunks[0]

Total Chunks: 55


'Revolutionizing Fashion Retail: Virtual Try-on\nEmpowered By Deep Learning\nPreethi S\nDepartment of Computer Science and Engineering\nCenter for Undergraduate and Postgraduate studies,BPUT\nRourkela, India\npreethiacharjya@gmail.com\nDebashreet Das\nDepartment of Computer Science and Engineering\nCenter for Undergraduate and Postgraduate studies,BPUT\nRourkela, India\ndebashreetdas12@gmail.com\nAbstract—Online clothing purchasing has become a\nwidespread practice for millions of individuals worldwide\nin the'

In [9]:
#recursive chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

recursive_chunks = splitter.split_text(text)

print("Total Recursive Chunks:", len(recursive_chunks))

recursive_chunks[0]


Total Recursive Chunks: 53


'Revolutionizing Fashion Retail: Virtual Try-on\nEmpowered By Deep Learning\nPreethi S\nDepartment of Computer Science and Engineering\nCenter for Undergraduate and Postgraduate studies,BPUT\nRourkela, India\npreethiacharjya@gmail.com\nDebashreet Das\nDepartment of Computer Science and Engineering\nCenter for Undergraduate and Postgraduate studies,BPUT\nRourkela, India\ndebashreetdas12@gmail.com\nAbstract—Online clothing purchasing has become a\nwidespread practice for millions of individuals worldwide'

In [ ]:
#semantic chunking
text2 = ""

pdf2 = PdfReader("papers/rbireport.pdf")


for page in pdf2.pages:
    extracted = page.extract_text()

    if extracted:
        text2 += extracted

semantic_chunks = re.split(
    r"(?i)(?=\bchapter\s+(?:[ivxlcdm]+|\d+)\b)",
    text2
)

semantic_chunks = [chunk.strip() for chunk in semantic_chunks if chunk.strip()]

print("Total Semantic Chunks:", len(semantic_chunks))

for i in range(5):
    print(f"\nSemantic Chunk {i+1}")
    print("-"*60)
    print(semantic_chunks[i])

# Retrieval Metrics in RAG

Retrieval metrics help us evaluate **how good our retriever is** at finding relevant documents or chunks for a user's query.

---

## Example

### Documents

| Document | Topic |
|----------|-------|
| D1 | Transformers |
| D2 | CNN |
| D3 | RAG |
| D4 | BERT |
| D5 | LLMs |

### Query

> What is Retrieval Augmented Generation?

### Relevant Documents

```text
D3
D5
```

### Retrieved Documents (Top 3)

```text
Rank 1 → D2
Rank 2 → D3
Rank 3 → D4
```

Only **D3** is both **retrieved** and **relevant**.

---

# 1. Precision@K

### Definition

Precision measures:

> **Out of the documents retrieved, how many are actually relevant?**

### Formula

\[
Precision@K = \frac{\text{Relevant Retrieved}}{\text{Retrieved}}
\]

### Example

Retrieved:

```text
D2
D3
D4
```

Relevant:

```text
D3
D5
```

Relevant Retrieved = **1** (only D3)

Retrieved = **3**

So,

```text
Precision@3 = 1 / 3 = 0.33
```

### Interpretation

- High Precision → Fewer irrelevant documents returned.
- Low Precision → Many irrelevant documents returned.

---

# 2. Recall@K

### Definition

Recall measures:

> **Out of all the relevant documents that exist, how many did we retrieve?**

### Formula

\[
Recall@K = \frac{\text{Relevant Retrieved}}{\text{Total Relevant}}
\]

### Example

Relevant Retrieved = **1** (D3)

Total Relevant = **2** (D3, D5)

```text
Recall@3 = 1 / 2 = 0.5
```

### Interpretation

- High Recall → Most relevant documents were found.
- Low Recall → Many relevant documents were missed.

---

# 3. MRR (Mean Reciprocal Rank)

### Definition

MRR measures:

> **How quickly was the first relevant document found?**

Unlike Precision and Recall, MRR only considers the **first relevant document**.

### Formula

\[
MRR = \frac{1}{\text{Rank of First Relevant Document}}
\]

### Example

Retrieved:

```text
Rank 1 → D2 ❌
Rank 2 → D3 ✅
Rank 3 → D4
```

The first relevant document is at **Rank 2**.

```text
MRR = 1 / 2 = 0.5
```

### Interpretation

- First relevant at Rank 1 → MRR = 1.0 (Best)
- First relevant at Rank 2 → MRR = 0.5
- First relevant at Rank 3 → MRR = 0.33
- No relevant document found → MRR = 0

---

# Comparison

| Metric | Measures | Formula |
|---------|----------|----------|
| Precision@K | How many retrieved documents are relevant | Relevant Retrieved / Retrieved |
| Recall@K | How many relevant documents were retrieved | Relevant Retrieved / Total Relevant |
| MRR | How early the first relevant document appears | 1 / Rank of First Relevant |

---

# Key Takeaways

- **Precision** focuses on the quality of retrieved results.
- **Recall** focuses on finding all relevant results.
- **MRR** focuses on how early the first useful result appears.

For **RAG systems**, **MRR is especially important** because LLMs perform best when the most relevant chunk appears near the top of the retrieved results.